### 0. Environment Setup
Install required libraries and download NLTK WordNet data for the synonym perturbation engine.

In [1]:
!pip install transformers datasets torch nltk pandas tqdm

import os
import re
import nltk
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.notebook import tqdm
from transformers import AutoModelForMaskedLM, AutoTokenizer

# Suppress Hugging Face warnings
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"

# Download WordNet for synonym substitution
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.corpus import wordnet as wn

### 1. Transformer (BERT) Scorer
The core Masked Language Model evaluator. This is identical to the baseline to ensure a mathematically valid 1:1 comparison.

In [2]:
class MaskedLMPLLScorer:
    def __init__(self, model_name: str, device: torch.device):
        self.model_name = model_name
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
        self.model.eval()

    @torch.no_grad()
    def sentence_pll(self, sentence: str) -> float:
        enc = self.tokenizer(sentence, return_tensors="pt")
        input_ids = enc["input_ids"].to(self.device)
        attention_mask = enc["attention_mask"].to(self.device)
        seq_len = input_ids.size(1)
        pll = 0.0
        special_mask = self.tokenizer.get_special_tokens_mask(input_ids[0].tolist(), already_has_special_tokens=True)

        for pos in range(seq_len):
            if special_mask[pos] == 1: continue
            original_token_id = input_ids[0, pos].item()
            masked_ids = input_ids.clone()
            masked_ids[0, pos] = self.tokenizer.mask_token_id
            outputs = self.model(input_ids=masked_ids, attention_mask=attention_mask)
            logits = outputs.logits[0, pos]
            log_probs = torch.log_softmax(logits, dim=-1)
            pll += float(log_probs[original_token_id].item())
        return pll

    def compare_candidates(self, candidate1: str, candidate2: str) -> str:
        score1 = self.sentence_pll(candidate1)
        score2 = self.sentence_pll(candidate2)
        return "1" if score1 >= score2 else "2"

### 2. The Novelty: Linguistic Perturbation Engine
These functions introduce controlled modifications to the sentences to test if the model's "reasoning" is robust or fragile.

In [3]:
ADJ_POLARITY_MAP = {
    "big": "small", "small": "big", "large": "small", "tiny": "large",
    "old": "young", "young": "old", "strong": "weak", "weak": "strong",
    "heavy": "light", "light": "heavy", "wide": "narrow", "narrow": "wide",
    "long": "short", "short": "long", "high": "low", "low": "high",
    "rich": "poor", "poor": "rich",
}

def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def synonym_substitution(sentence: str) -> Optional[str]:
    tokens = re.findall(r"\w+|[^\w\s]", sentence)
    for i, tok in enumerate(tokens):
        if not re.match(r"^[A-Za-z]+$", tok): continue
        lower = tok.lower()
        synsets = wn.synsets(lower)
        candidates = []
        for syn in synsets:
            for lemma in syn.lemmas():
                name = lemma.name().replace("_", " ")
                if name.lower() != lower and " " not in name and name.isalpha():
                    candidates.append(name)
        if candidates:
            replacement = candidates[0].capitalize() if tok[0].isupper() else candidates[0]
            new_tokens = tokens[:]
            new_tokens[i] = replacement
            out = "".join([(t if re.match(r"[^\w\s]", t) else (" " + t)) for t in new_tokens]).strip()
            return normalize_space(out)
    return None

def adjective_polarity_change(sentence: str) -> Optional[str]:
    for src, tgt in ADJ_POLARITY_MAP.items():
        pattern = re.compile(rf"\b{re.escape(src)}\b", re.IGNORECASE)
        match = pattern.search(sentence)
        if match:
            matched = match.group(0)
            replacement = tgt.capitalize() if matched[0].isupper() else tgt
            return pattern.sub(replacement, sentence, count=1)
    return None

def apply_perturbations(sentence: str) -> Dict[str, str]:
    outputs = {}
    syn = synonym_substitution(sentence)
    if syn and syn != sentence: outputs["synonym"] = syn
    
    pol = adjective_polarity_change(sentence)
    if pol and pol != sentence: outputs["polarity"] = pol
    
    return outputs

### 3. Data Loading
Strictly extracts the first 300 sequential sentences from the WinoGrande validation set to match the baseline.

In [4]:
@dataclass
class Example:
    sentence: str
    option1: str
    option2: str
    answer: str
    example_id: str

def replace_blank(sentence: str, replacement: str) -> str:
    return normalize_space(sentence.replace("_", replacement, 1))

def load_winogrande_examples(split: str, max_items: int) -> List[Example]:
    ds = load_dataset("winogrande", "winogrande_xl", split=split)
    rows = list(ds)[:max_items]
    examples = []
    for idx, row in enumerate(rows):
        row_id = str(row.get("qID", row.get("id", idx)))
        examples.append(Example(
            sentence=row["sentence"], option1=row["option1"], 
            option2=row["option2"], answer=str(row["answer"]), example_id=row_id
        ))
    return examples

### 4. Main Experiment Execution
Evaluates the base accuracy, applies perturbations, recalculates accuracy, and flags flipped predictions.

In [5]:
def run_main_experiment():
    MODEL_NAME = "bert-base-uncased"
    MAX_ITEMS = 300 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    scorer = MaskedLMPLLScorer(model_name=MODEL_NAME, device=device)
    examples = load_winogrande_examples("validation", MAX_ITEMS)
    print(f"Loaded the first {len(examples)} examples from WinoGrande.\n")
    
    results = []
    
    for ex in tqdm(examples, desc="Evaluating Perturbations"):
        # 1. Base Evaluation
        cand1 = replace_blank(ex.sentence, ex.option1)
        cand2 = replace_blank(ex.sentence, ex.option2)
        base_pred = scorer.compare_candidates(cand1, cand2)
        base_correct = (base_pred == ex.answer)
        
        # 2. Apply Perturbations
        perturbed_versions = apply_perturbations(ex.sentence)
        
        for p_type, p_sentence in perturbed_versions.items():
            try:
                p_cand1 = replace_blank(p_sentence, ex.option1)
                p_cand2 = replace_blank(p_sentence, ex.option2)
                p_pred = scorer.compare_candidates(p_cand1, p_cand2)
                p_correct = (p_pred == ex.answer)
                
                results.append({
                    "example_id": ex.example_id,
                    "perturbation_type": p_type,
                    "original_sentence": ex.sentence,
                    "perturbed_sentence": p_sentence,
                    "gold_answer": ex.answer,
                    "original_correct": base_correct,
                    "perturbed_correct": p_correct,
                    "changed_prediction": (base_pred != p_pred)
                })
            except Exception:
                continue # Skip if perturbation broke the placeholder logic
                
    df_results = pd.DataFrame(results)
    
    # 5. Metric Aggregation to Highlight Novelty
    print("\n--- MAIN EXPERIMENT RESULTS ---")
    summary = df_results.groupby("perturbation_type").agg(
        total_tested=("perturbation_type", "count"),
        original_accuracy=("original_correct", "mean"),
        perturbed_accuracy=("perturbed_correct", "mean"),
        changed_prediction_rate=("changed_prediction", "mean")
    ).reset_index()
    
    # Format as percentages for cleaner viewing
    summary['original_accuracy'] = (summary['original_accuracy'] * 100).map('{:.2f}%'.format)
    summary['perturbed_accuracy'] = (summary['perturbed_accuracy'] * 100).map('{:.2f}%'.format)
    summary['changed_prediction_rate'] = (summary['changed_prediction_rate'] * 100).map('{:.2f}%'.format)
    
    print(summary.to_string(index=False))
    
    return df_results, summary

raw_df, summary_df = run_main_experiment()
raw_df[raw_df['changed_prediction'] == True].head() # Show examples where the model broke

Using device: cpu


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded the first 300 examples from WinoGrande.



Evaluating Perturbations:   0%|          | 0/300 [00:00<?, ?it/s]


--- MAIN EXPERIMENT RESULTS ---
perturbation_type  total_tested original_accuracy perturbed_accuracy changed_prediction_rate
         polarity            36            41.67%             41.67%                   0.00%
          synonym           300            50.00%             50.00%                  16.67%


,example_id,perturbation_type,original_sentence,perturbed_sentence,gold_answer,original_correct,perturbed_correct,changed_prediction
26,22,synonym,"Cynthia violated the rights of Amy, because _ ...","Artemis violated the rights of Amy, because _ ...",2,False,True,True
32,27,synonym,The portions of food today were bigger than th...,The part of food today were bigger than the si...,1,True,False,True
36,31,synonym,Mary thought poodles were a cool dog but Rache...,Madonna thought poodles were a cool dog but Ra...,1,True,False,True
38,32,synonym,Mary thought poodles were a cool dog but Rache...,Madonna thought poodles were a cool dog but Ra...,2,False,True,True
54,45,synonym,Since Craig wears clear contacts and William w...,Since Craig wear clear contacts and William we...,1,True,False,True


In [6]:
# Filter the results for only the polarity perturbations
polarity_df = raw_df[raw_df['perturbation_type'] == 'polarity']

print(f"Total Polarity Sentences Found: {len(polarity_df)}\n")

# Loop through and print the original vs. perturbed sentences
for index, row in polarity_df.iterrows():
    print(f"Original:  {row['original_sentence']}")
    print(f"Perturbed: {row['perturbed_sentence']}")
    print("-" * 60)

Total Polarity Sentences Found: 36

Original:  Terry tried to bake the eggplant in the toaster oven but the _ was too big.
Perturbed: Terry tried to bake the eggplant in the toaster oven but the _ was too small.
------------------------------------------------------------
Original:  Natalie has a rich husband and lots of money, Jennifer is poor _ needs to make her clothes.
Perturbed: Natalie has a poor husband and lots of money, Jennifer is poor _ needs to make her clothes.
------------------------------------------------------------
Original:  I had to read an entire story for class tomorrow. Luckily, the _ was short.
Perturbed: I had to read an entire story for class tomorrow. Luckily, the _ was long.
------------------------------------------------------------
Original:  Michael just bought brand new wheels for his truck unlike Leslie because _ wheels were old and used.
Perturbed: Michael just bought brand new wheels for his truck unlike Leslie because _ wheels were young and used.
